In [ ]:
from dataclasses import dataclass
import torch


class ConfSorter:
    pass
# end

class TruthSorter(ConfSorter):
    pass
# end

class MaxSorter(ConfSorter):
    pass
# end

class RandomSorter(ConfSorter):
    pass
# end

sorter = TruthSorter()
sorter = MaxSorter()
sorter = RandomSorter()

class LlaDAModel():
    pass
# end

class DiffusionStepHelper():
    pass
# end

class BlockDiffusionStepHelper(DiffusionStepHelper):
    pass
# end

@dataclass
class DiffusionConfig:
    num_blocks: int
# end


model = LlaDAModel()
model.enable_past_kv()

# 处理past_key_values
def run_model_with_snapshot(model, sorter, config_diffusion):

    idx_refresh = torch.tensor([[]])

    for id_block in range(num_blocks):
        step_helper = StepHelper(step_per_block, size_block)    # ???

        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask = x[:,:position_end] == id_mask

        idx_denoising = x[:, position_start:position_end]
        x_current = torch.cat([idx_refresh, idx_denoising], dim=-1) 

        logits = model(x_current, idx_refresh=idx_refresh, idx_denoising=idx_denoising)

        snapshot = LLaDAOutputSnapshot(logits)\
                    .set_mask_id(id_mask)\
                    .transform_logits(mask_mask)

        idx_sorted_by_conf = snapshot.generate_sorted_sequence(sorter)
        
        num_unmask = step_helper.get_step()
        idx_unmask = idx_sorted_by_conf[:, :num_unmask]
        snapshot.unmask_by_idx(idx_unmask)  # get the x0, x0_p

        idx_refresh = idx_unmask

        for step in range(1, step_per_block):
            idx_sorted_by_conf = snapshot.generate_sorted_sequence(sorter)
            num_unmask = step_helper.get_step()
            idx_unmask = idx_sorted_by_conf[:, :num_unmask]
            idx_denoising = idx_unmask
            idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1)   # 原本的位置

            x_current = x[:, idx_current]
            logits = model(x_current, idx_current=idx_current) # 有问题兄弟

            logits_refresh = logits[:,:idx_refresh.shape[-1]]
            logits_denoising = logits[:, idx_denoising.shape[-1]:]

            snapshot.update_logits(logits_refresh, idx_refresh)
            snapshot.unmask_by_logits(logits_denoising, idx_denoising)

            idx_refresh = idx_denoising
        # end


    # end
# end